<h1>Structure Sentinel-3 quality-masked, reprojected GeoTIFFs into the n x N (spatial grid x time) data matrices required by the SS-FPCA (Gong et al. 2021)</h1>

<p>Steps:<br>
1. Read all GeoTIFFs (previously reprojected to MSLC2000) in the input folder and group by variable and date<br>
2. Read the CRS, transform, and shape of all GeoTIFFs to build a union bounding box/spatial extent<br>
3. For each variable, composite same-day granules by taking the per-pixel mean of valid values for that day<br>
4. Assemble the n x N matrix, where n = valid pixels for that day, N = number of days<br>
5. Save three CSV files per variable: n x N data matrix, pixel coordinates, and metadata<br>
6. (Optional) Export some composited daily images as GeoTIFFs and PNG maps for visual checking<p>

In [1]:
import os
import re
import glob
import warnings
from datetime import datetime
from collections import defaultdict
 
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
 
warnings.filterwarnings('ignore') # supresses warning messages

In [2]:
# CONFIGURATION
# need to edit these parameters

# expects GeoTIFFs with naming convention:
# <granule_name>_qualitymasked_CHL_NN.tif
# <granule_name>_qualitymasked_CHL_OC4ME.tif
GEOTIFF_DIR = '/Users/gwyneth/Desktop/oct2022_mar2023_geotiffs'

# output directory for CSVs and optional spot-check exports
OUTPUT_DIR = '/Users/gwyneth/Desktop/oct2022_mar2023_ssfpca'

# variables to process
# keys must match the suffix in GeoTIFF filenames (case-sensitive)
VARIABLES = {
    'CHL_NN': 'chl_nn',
    'CHL_OC4ME': 'chl_oc4me',
}

# number of composited daily GeoTIFFs and PNG maps to export for checking
# outputs are chosen as the first N days with any valid data
# set to 0 to skip
SPOT_CHECK_N = 2

# name stem used for all output files
OUTPUT_STEM = 'ssfpca_data'

In [3]:
# for daily composite, need to parse the sensing date from the GeoTIFF filenames

def extract_date(filepath):

    basename = os.path.basename(filepath)
    m = re.search(r'(\d{8})T\d{6}', basename)
    if m:
        return datetime.strptime(m.group(1), '%Y%m%d').date()
    return None

# scans the spatial extent of all GeoTIFFs to build a union bounding box
# returns a common affine transform and canvas shape (height, width) that covers all granules at the natve pixel size of the first GeoTIFF
def build_common_canvas(all_tif_paths):

    x_mins, y_mins, x_maxs, y_maxs = [], [], [], []
    crs = None
    pixel_size = None
 
    for tif_path in all_tif_paths:
        with rasterio.open(tif_path) as src:
            bounds = src.bounds # (left, bottom, right, top)
            x_mins.append(bounds.left)
            y_mins.append(bounds.bottom)
            x_maxs.append(bounds.right)
            y_maxs.append(bounds.top)
            if crs is None:
                crs = src.crs
                pixel_size = src.transform.a # pixel width (positive)
 
    # union bounding box
    x_min = min(x_mins)
    y_min = min(y_mins)
    x_max = max(x_maxs)
    y_max = max(y_maxs)
 
    W = int(round((x_max - x_min) / pixel_size))
    H = int(round((y_max - y_min) / pixel_size))
 
    # from_origin: top-left corner, pixel width, pixel height (positive)
    common_transform = from_origin(x_min, y_max, pixel_size, pixel_size)
 
    print(f"\nCommon canvas (union of all granule extents):")
    print(f"CRS: {crs}")
    print(f"Pixel size: {pixel_size:.1f} m")
    print(f"Extent: x=[{x_min:.0f}, {x_max:.0f}]  "
          f"y=[{y_min:.0f}, {y_max:.0f}]")
    print(f"Size: {W} cols x {H} rows ({W*H:,} pixels)")
 
    return common_transform, H, W, crs, pixel_size

# reproject each GeoTIFF onto the common canvas
# both source and output are in MSLC2000, so this is just pixel alignment with no CRS transformation

# this function fixes varying origins from input GeoTIFFs too
# because every granule is reprojected onto the same common canvas with the same transform before accumulation, the pixel alignment is enforced by rasterio
 
def reproject_to_canvas(tif_path, common_transform, H, W, crs):

    with rasterio.open(tif_path) as src:
        src_data = src.read(1).astype(np.float32)
        src_nodata = src.nodata
        src_transform = src.transform
        src_crs = src.crs
 
    # normalise nodata to NaN in the source array
    if src_nodata is not None and not np.isnan(src_nodata):
        src_data[src_data == src_nodata] = np.nan
 
    # use a finite nodata for the reproject call (rasterio requirement)
    NODATA_SENTINEL = -9999.0
    src_data_nd = np.where(np.isfinite(src_data), src_data, NODATA_SENTINEL)
 
    canvas = np.full((H, W), NODATA_SENTINEL, dtype=np.float32)
 
    reproject(
        source         = src_data_nd,
        destination    = canvas,
        src_transform  = src_transform,
        src_crs        = src_crs,
        dst_transform  = common_transform,
        dst_crs        = crs,
        src_nodata     = NODATA_SENTINEL,
        dst_nodata     = NODATA_SENTINEL,
        resampling     = Resampling.nearest, # same CRS: nearest preserves values
    )
 
    canvas[canvas == NODATA_SENTINEL] = np.nan
    return canvas

# export a composited daily image as GeoTIFF and PNG for spot-checking

def export_spot_check(daily_grid, common_transform, H, W, crs,
                      date, varname, out_dir):

    check_dir = os.path.join(out_dir, 'spot_checks')
    os.makedirs(check_dir, exist_ok=True)
    stem = f'{OUTPUT_STEM}_{varname}_{date}'
 
    # GeoTIFF
    tif_path = os.path.join(check_dir, f'{stem}.tif')
    with rasterio.open(
        tif_path, 'w',
        driver    = 'GTiff',
        height    = H,
        width     = W,
        count     = 1,
        dtype     = np.float32,
        crs       = crs,
        transform = common_transform,
        nodata    = np.nan,
        compress  = 'lzw',
    ) as dst:
        dst.write(daily_grid[np.newaxis, :, :])
 
    # PNG map
    png_path = os.path.join(check_dir, f'{stem}.png')
    valid = daily_grid[np.isfinite(daily_grid)]
    vmin  = np.percentile(valid, 2)  if valid.size > 0 else 0
    vmax  = np.percentile(valid, 98) if valid.size > 0 else 1
 
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(daily_grid, origin='upper', cmap='viridis',
                   vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label=varname)
    ax.set_title(f'{varname}  —  {date}\n'
                 f'Composited daily mean (EPSG:5479 MSLC2000)')
    ax.set_xlabel('Column index (projected x)')
    ax.set_ylabel('Row index (projected y, top = north)')
    plt.tight_layout()
    plt.savefig(png_path, dpi=120)
    plt.close()
 
    n_valid = int(np.sum(np.isfinite(daily_grid)))
    print(f'GeoTIFF: {tif_path}')
    print(f'PNG: {png_path}')
    print(f'Valid px: {n_valid:,} / {H*W:,} ({100*n_valid/(H*W):.1f} %)')

# main pipeline
# putting everything together

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    # 1. Read GeoTIFFs and group by variable then date
    all_tifs = sorted(glob.glob(os.path.join(GEOTIFF_DIR, '*.tif')))
    if not all_tifs:
        raise FileNotFoundError(
            f"No .tif files found in '{GEOTIFF_DIR}'. "
            "Edit GEOTIFF_DIR in the CONFIG block.")
 
    print(f"Found {len(all_tifs)} GeoTIFF(s) in '{GEOTIFF_DIR}'.")
 
    # var_date_files[suffix][date] = [list of tif paths]
    var_date_files = {v: defaultdict(list) for v in VARIABLES}
    skipped = 0
 
    for tif_path in all_tifs:
        basename = os.path.basename(tif_path)
        matched_var = None
        for suffix in VARIABLES:
            if basename.endswith(f'_{suffix}.tif'):
                matched_var = suffix
                break
        if matched_var is None:
            skipped += 1
            continue
 
        date = extract_date(tif_path)
        if date is None:
            print(f'Warning: Cannot parse date from {basename}, skipping.')
            skipped += 1
            continue
 
        var_date_files[matched_var][date].append(tif_path)
 
    if skipped:
        print(f"{skipped} file(s) skipped (unrecognised suffix or no date).")
 
    # collect all matched tif paths for canvas computation
    all_matched = [p for var in VARIABLES
                     for paths in var_date_files[var].values()
                     for p in paths]
    if not all_matched:
        raise RuntimeError("No usable GeoTIFFs found after filtering.")
 
    # 2. Build the common canvas from the union of all granule extents
    common_transform, H, W, crs, pixel_size = build_common_canvas(all_matched)
 
    # pixel centre coordinates in the projected CRS
    x_coords = common_transform.c + (np.arange(W) + 0.5) * common_transform.a
    y_coords = common_transform.f + (np.arange(H) + 0.5) * common_transform.e
 
    # 3. Process each variable
    for suffix, varname in VARIABLES.items():
        date_files = var_date_files[suffix]
        if not date_files:
            print(f"\nSkipping: No files found for variable '{suffix}'.")
            continue
 
        dates_sorted = sorted(date_files.keys())
        N = len(dates_sorted)
        print(f"\nVariable: {varname} | ({N} day(s))")
 
        daily_grids = np.full((N, H, W), np.nan, dtype=np.float32)
        spot_checks_done = 0
 
        for t, date in enumerate(tqdm(dates_sorted,
                                      desc=f'Compositing {varname}')):
            tif_paths = date_files[date]
            accum_sum   = np.zeros((H, W), dtype=np.float64)
            accum_count = np.zeros((H, W), dtype=np.int32)
 
            for tif_path in tif_paths:
                try:
                    canvas = reproject_to_canvas(
                        tif_path, common_transform, H, W, crs)
                except Exception as e:
                    print(f'\nWarning: Could not reproject '
                          f'{os.path.basename(tif_path)}: {e}')
                    continue
 
                valid = np.isfinite(canvas)
                accum_sum[valid] += canvas[valid]
                accum_count[valid] += 1
 
            observed = accum_count > 0
            daily_grids[t][observed] = (
                accum_sum[observed] / accum_count[observed]
            ).astype(np.float32)
 
            # spot-check exports
            if spot_checks_done < SPOT_CHECK_N and observed.any():
                print(f'\n  Spot-check: {varname}  {date}')
                export_spot_check(daily_grids[t], common_transform, H, W, crs,
                                  date, varname, OUTPUT_DIR)
                spot_checks_done += 1
 
        # 4. Masking for pixels observed at least once
        ocean_mask = np.any(np.isfinite(daily_grids), axis=0)
        n_ocean = int(ocean_mask.sum())
        print(f"\nValid pixels (ocean mask): {n_ocean:,} / {H*W:,} "
              f"({100*n_ocean/(H*W):.1f} %)")
 
        if n_ocean == 0:
            print(f"ERROR: No valid pixels for {varname}.")
            continue
 
        # 5. Assemble n x N data matrix
        data_matrix = np.full((n_ocean, N), np.nan, dtype=np.float32)
        for t in range(N):
            data_matrix[:, t] = daily_grids[t][ocean_mask]
 
        n_obs = int(np.sum(np.isfinite(data_matrix)))
        miss_pct = 100.0 * (1 - n_obs / (n_ocean * N))
        print(f"Data matrix shape: {data_matrix.shape} (n x N)")
        print(f"Overall missing: {miss_pct:.1f} %")
        print(f"Value range: [{np.nanmin(data_matrix):.4f}, "
              f"{np.nanmax(data_matrix):.4f}]")
 
        # 6. Save CSV outputs
        row_idx, col_idx = np.where(ocean_mask)
        px_x = x_coords[col_idx]
        px_y = y_coords[row_idx]
        dates_str = [str(d) for d in dates_sorted]
 
        # (a) Data matrix
        data_path = os.path.join(OUTPUT_DIR,
                                 f'{OUTPUT_STEM}_{varname}_data_matrix.csv')
        pd.DataFrame(data_matrix, columns=dates_str).to_csv(
            data_path, index=False, na_rep='NA')
        print(f"Saved data matrix  : {data_path}")
 
        # (b) Pixel coordinates
        coords_path = os.path.join(OUTPUT_DIR,
                                   f'{OUTPUT_STEM}_{varname}_pixel_coords.csv')
        pd.DataFrame({
            'px_x': px_x,
            'px_y': px_y,
            'row_idx': row_idx,
            'col_idx': col_idx,
        }).to_csv(coords_path, index=False)
        print(f"Saved pixel coords : {coords_path}")
 
        # (c) Grid metadata
        meta_path = os.path.join(OUTPUT_DIR,
                                 f'{OUTPUT_STEM}_{varname}_metadata.csv')
        pd.DataFrame({
            'grid_height': [H],
            'grid_width': [W],
            'pixel_size_m': [pixel_size],
            'epsg': [crs.to_epsg()],
        }).to_csv(meta_path, index=False)
        print(f"Saved metadata: {meta_path}")
 
    print("\n=== Done ===")

# run the pipeline
main()

Found 281 GeoTIFF(s) in '/Users/gwyneth/Desktop/oct2022_mar2023_geotiffs'.

Common canvas (union of all granule extents):
CRS: EPSG:5479
Pixel size: 299.7 m
Extent: x=[6935283, 7147936]  y=[4943072, 5156231]
Size: 710 cols x 711 rows (504,810 pixels)

Variable: chl_nn | (26 day(s))


Compositing chl_nn:   0%|          | 0/26 [00:00<?, ?it/s]


  Spot-check: chl_nn  2022-11-19


Compositing chl_nn:   4%|▍         | 1/26 [00:00<00:08,  2.80it/s]

GeoTIFF: /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/spot_checks/ssfpca_data_chl_nn_2022-11-19.tif
PNG: /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/spot_checks/ssfpca_data_chl_nn_2022-11-19.png
Valid px: 4 / 504,810 (0.0 %)

  Spot-check: chl_nn  2022-12-01


...

GeoTIFF: /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/spot_checks/ssfpca_data_chl_oc4me_2022-10-29.tif
PNG: /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/spot_checks/ssfpca_data_chl_oc4me_2022-10-29.png
Valid px: 13 / 504,810 (0.0 %)

  Spot-check: chl_oc4me  2022-10-30


Compositing chl_oc4me:   7%|▋         | 6/88 [00:00<00:06, 12.76it/s]

GeoTIFF: /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/spot_checks/ssfpca_data_chl_oc4me_2022-10-30.tif
PNG: /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/spot_checks/ssfpca_data_chl_oc4me_2022-10-30.png
Valid px: 5,451 / 504,810 (1.1 %)


Compositing chl_oc4me: 100%|██████████| 88/88 [00:05<00:00, 17.36it/s]



Valid pixels (ocean mask): 48,634 / 504,810 (9.6 %)
Data matrix shape: (48634, 88) (n x N)
Overall missing: 91.9 %
Value range: [-2.0000, 1.9843]
Saved data matrix  : /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/ssfpca_data_chl_oc4me_data_matrix.csv
Saved pixel coords : /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/ssfpca_data_chl_oc4me_pixel_coords.csv
Saved metadata: /Users/gwyneth/Desktop/oct2022_mar2023_ssfpca/ssfpca_data_chl_oc4me_metadata.csv

=== Done ===
